# 01 - Análisis Exploratorio y Selección de Modelo (Riesgo Crediticio)

Este notebook documenta la exploración del dataset de riesgo crediticio, el diseño del `ColumnTransformer` de preprocesamiento, la comparación de modelos candidatos mediante validación cruzada estratificada y una simulación local de detección de **Data Drift**.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.src.data import load_or_create_dataset
from model.src.schema import FEATURE_NAMES, TARGET_COLUMN
from model.src.train import build_preprocessor, get_candidate_models

## 1. Carga y exploración inicial de datos

In [ ]:
data_path = PROJECT_ROOT / "model" / "data" / "credit_risk.csv"
df = load_or_create_dataset(data_path)

print(f"Shape: {df.shape}")
print(f"Tasa de default: {df[TARGET_COLUMN].mean():.2%}")
print("\nValores nulos por columna:")
print(df.isna().sum())
df.head()

In [ ]:
df[FEATURE_NAMES].describe().T[['mean', 'std', 'min', '50%', 'max']]

## 2. Distribuciones y correlaciones

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), FEATURE_NAMES):
    for label, name in [(0, "Approved"), (1, "Default")]:
        subset = df.loc[df[TARGET_COLUMN] == label, col].dropna()
        ax.hist(subset, bins=25, alpha=0.55, density=True, label=name)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3. Comparación de modelos candidatos con Stratified K-Fold CV

In [ ]:
X = df[FEATURE_NAMES]
y = df[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, estimator in get_candidate_models(random_state=42).items():
    pipe = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", estimator),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    results.append({"model": name, "roc_auc_mean": scores.mean(), "roc_auc_std": scores.std()})

pd.DataFrame(results).sort_values("roc_auc_mean", ascending=False)

## 4. Evaluación del modelo ganador sobre holdout

In [ ]:
winner = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", get_candidate_models(random_state=42)["gradient_boosting"]),
])
winner.fit(X_train, y_train)

y_proba = winner.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["approved", "default"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0])
axes[0].set_title("Curva ROC (Holdout)")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["approved", "default"], ax=axes[1]
)
axes[1].set_title("Matriz de Confusión")
plt.tight_layout()
plt.show()